In [1]:
import numpy as np
from scipy.spatial.transform import Rotation
import plotly.graph_objects as go
import opengate as gate
import qmirt_utility as qmirt
import pandas as pd
from pathlib import Path
import pyvista as pv
import os
# 1. 必须在导入 pyvista 之前，强行设置环境变量使用 CPU 软件渲染
os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"

pv.OFF_SCREEN = True

# 3. 尝试启动虚拟显示（这会解决底层 vtkXOpenGLRenderWindow 的报错）
try:
    pv.start_xvfb()
except Exception:
    pass

In [4]:
import warnings
warnings.filterwarnings('ignore')

import os
os.environ['VTK_SILENCE_GET_CELL_SIZE_WARNING'] = '1'

def extract_solid_names_from_wrl(wrl_path):
    """Extract SOLID names from WRL file comments"""
    solid_names = []
    try:
        with open(wrl_path, 'r') as f:
            for line in f:
                if '#---------- SOLID:' in line:
                    # Extract the name after "SOLID: "
                    name = line.split('#---------- SOLID:')[1].strip()
                    solid_names.append(name)
    except Exception as e:
        print(f"Error reading WRL file: {e}")
    return solid_names

def load_wrl_as_mesh(wrl_path):
    import vtk
    import pyvista as pv
    import logging
    
    # 抑制所有 VTK 和 PyVista 警告
    vtk.vtkObject.GlobalWarningDisplayOff()
    logging.disable(logging.CRITICAL)
    
    # 1. 使用底层的 VTK Importer 读取 WRL 场景
    importer = vtk.vtkVRMLImporter()
    importer.SetFileName(wrl_path)
    importer.Update()  # 【修复 1】：Importer 的执行方法是 Update()
    
    # 创建一个空的 AppendPolyData 用于合并 WRL 里的所有几何组件
    append_filter = vtk.vtkAppendPolyData()
    
    # 2. 遍历 WRL 场景里的所有渲染器和 Actor，提取网格
    renderer = importer.GetRenderer()
    actors = renderer.GetActors()
    actors.InitTraversal()
    
    for i in range(actors.GetNumberOfItems()):
        actor = actors.GetNextActor()
        if actor and actor.GetMapper():
            poly_data = actor.GetMapper().GetInput()
            if poly_data:
                # 【修复 2】：纯 VTK 对象，必须使用大写 AddInputData
                append_filter.AddInputData(poly_data) 
                
    # 【修复 3】：纯 VTK 过滤器，执行更新也必须用大写 Update()
    append_filter.Update()
    
    # 【修复 4】：获取结果必须用大写 GetOutput()，然后再交给 pyvista 包装
    mesh = pv.wrap(append_filter.GetOutput())
    
    return mesh




Total SOLID objects in WRL: 2081


In [7]:

# 从 WRL 文件直接提取原始对象名字
wrl_path = "/home/fanghan/Work/RPIL/QMIRT/gate10mc/dev/dc_spect_geometry.wrl"
original_solid_names = extract_solid_names_from_wrl(wrl_path)

# 加载 WRL 文件并获取几何体
detector_mesh = load_wrl_as_mesh(wrl_path)

# print("=== WRL 文件中的原始 SOLID 对象名字 ===")
# for i, name in enumerate(original_solid_names):
#     print(f"  [{i}] {name}")

print(f"\nTotal SOLID objects in WRL: {len(original_solid_names)}")


Total SOLID objects in WRL: 50081


In [8]:
# Draw the detected mesh using pyvista and save to html
plotter = pv.Plotter(off_screen=True)
plotter.add_mesh(detector_mesh, color='lightblue', show_edges=True)
html_path = "dc_spect_geometry.html"
plotter.export_html(html_path)
